# Entrenamiento del árbol de decisión — original y enriquecido

El algoritmo original y su entrenamiento permanecen en `decision-tree.ipynb`, sin cambios.
Ejecutá las celdas de arriba hacia abajo con Python 3. Las celdas del algoritmo
sólo necesitan la biblioteca estándar; el panel final requiere `ipywidgets`.
Este notebook ejecuta solamente el entrenamiento enriquecido, con continuos, nulos
y poda mediante APIs separadas en español. No importa ni reentrena el modelo original.
Los tres ítems de revisión fueron aprobados explícitamente por el usuario.

### Cómo leer las referencias

Se distingue la **página impresa del libro** de la **página dentro del archivo PDF**.
Estas abreviaturas identifican los archivos que proporcionaste:

| Referencia | Archivo PDF | Páginas del libro | Páginas del archivo |
| --- | --- | --- | --- |
| CON | `(atributos continuos) Quinlan - C4_5_ Programs for Machine Learning.pdf` | 25–26 | 1–2 |
| NUL | `(valores nulos) Quinlan - C4_5_ Programs for Machine Learning.pdf` | 27–33 | 1–7 |
| POD | `(poda)C4_5_ Programs for Machine Learning.pdf` | 37–43 | 1–7 |

**Fuente conceptual no significa código transcrito.** El libro nombra `info(T)`,
`info_X(T)`, `gain(X)`, `split info(X)` y `U_CF(E,N)`; no define una función Python
por cada auxiliar. Se usa esa notación cuando existe correspondencia directa y nombres
descriptivos en español para el resto. Las firmas y estructuras de datos son propias.
Los nombres de las funciones base copiadas se mantienen para no modificar el entrenamiento.


## 1. Datos de entrada

Proporcioná una matriz rectangular de listas, no vacía. La primera fila contiene los
nombres de las columnas; cada fila siguiente es una observación y la **última columna
siempre contiene la clase**. El encabezado no se pasa al entrenamiento. Los atributos
y clases deben poder usarse como claves de un diccionario; la base no valida la entrada.

El entrenamiento base divide todos los atributos por sus valores observados, incluidos
los números. Por ejemplo, `[1, "no"], [2, "no"], [3, "yes"], [4, "yes"]` produce ramas
para `1`, `2`, `3` y `4`, no un corte numérico en `2.5`.

Podés convertir entradas NumPy/pandas con `array.tolist()` o
`dataframe.to_numpy().tolist()`, proporcionando el encabezado por separado.

**Procedencia:** matriz y preparación copiadas de `decision-tree.ipynb`, no extraídas
de los PDFs. NUL, libro pp. 30–32 / PDF pp. 4–6, contiene otro ejemplo Play/Don't Play;
no se presenta esta matriz categórica como una transcripción de ese ejemplo.


In [ ]:
training_matrix = [  # Define D como una matriz con encabezados y la clase en la última columna.
    ["Outlook", "Temperature", "Humidity", "Wind", "Play"],  # Etiqueta las columnas de atributos y la clase objetivo Play.
    ["sunny", "hot", "high", "weak", "no"],  
    ["sunny", "hot", "high", "strong", "no"],  
    ["overcast", "hot", "high", "weak", "yes"],  
    ["rain", "mild", "high", "weak", "yes"],  
    ["rain", "cool", "normal", "weak", "yes"],  
    ["rain", "cool", "normal", "strong", "no"],  
    ["overcast", "cool", "normal", "strong", "yes"],  
    ["sunny", "mild", "high", "weak", "no"],  
    ["sunny", "cool", "normal", "weak", "yes"],  
    ["rain", "mild", "normal", "weak", "yes"],  
    ["sunny", "mild", "normal", "strong", "yes"],  
    ["overcast", "mild", "high", "strong", "yes"],  
    ["overcast", "hot", "normal", "weak", "yes"],  
    ["rain", "mild", "high", "strong", "no"],  
]  # Completa la matriz de ejemplos de entrenamiento.

columns = training_matrix[0]  # Separa los nombres de las columnas, que no son observaciones.
D = training_matrix[1:]  # Obtiene el conjunto D que recibe el algoritmo de la imagen.
A = list(range(len(columns) - 1))  # Forma A con los índices de atributos; excluye la columna de clase.


## 2. Estructura del árbol

Una hoja tiene `attribute=None` y devuelve `prediction`. Un nodo interno guarda el
índice del atributo y sus hijos, identificados por los valores observados.
El campo opcional `Node.threshold` está disponible para predicción e impresión,
pero el entrenamiento base siempre lo deja en `None`.

Cada nodo conserva su clase mayoritaria como respaldo ante categorías no vistas.
En caso de empate, se usa la primera clase encontrada.

**Procedencia:** la clase `Node` está copiada de `decision-tree.ipynb`; los PDFs no
definen esta clase Python. CON, libro pp. 25–26 / PDF pp. 1–2, fundamenta los umbrales.
NUL, libro pp. 30–32 / PDF pp. 4–6, describe distribuciones en las hojas: todavía no se
guardan en este `Node` base. El respaldo para categorías nuevas es comportamiento propio.


In [ ]:
class Node:  # Define la estructura del nodo T del pseudocódigo.
    def __init__(self, prediction, attribute=None, threshold=None):  # Inicializa un nodo como hoja; luego puede convertirse en nodo de decisión.
        self.prediction = prediction  # L2, L4 y L13: guarda la clase que devolvería esta hoja.
        self.attribute = attribute  # L15: atributo de decisión; None indica que el nodo es una hoja.
        self.threshold = threshold  # Campo opcional de los auxiliares; el entrenamiento literal no usa cortes.
        self.children = {}  # L19: almacena las ramas y sus subárboles hijos.


## 3. Evaluación de impureza

Los nombres del pseudocódigo original `impurityEval-1(D)` e `impurityEval-2(Ai,D)`
se representan como `impurityEval_1` e `impurityEval_2`, porque Python no permite guiones
en identificadores. Estos nombres base se conservan para mantener intacto el entrenamiento.

- **Antes de dividir:** `p0 = H(D) = -sum(p(c) * log2(p(c)))`.
- **Después de dividir:** `pi = sum(|Dj| / |D| * H(Dj))`.
- **Ganancia de información:** `p0 - pi`.

Los auxiliares base asignan entropía cero al conjunto vacío. Un atributo constante tiene
ganancia cero. `partition()` e `impurityEval_2()` aceptan un `threshold` opcional como
utilidad independiente, pero `decision_tree()` nunca lo proporciona.

**Procedencia por función:**

| Función base | Origen y relación con los PDFs |
| --- | --- |
| `impurityEval_1` | Copiada del notebook original; NUL, libro pp. 28–30 / PDF pp. 2–4, utiliza `info(T)` y ejemplifica la entropía. No incluye estos nombres Python. |
| `partition` | Copiada del original; CON, libro pp. 25–26 / PDF pp. 1–2, describe los cortes, y NUL, libro p. 29 / PDF p. 3, la partición fraccionaria que esta función base **no** realiza. |
| `impurityEval_2` | Copiada del original; NUL, libro pp. 28–30 / PDF pp. 2–4, utiliza `info_X(T)`. La base cuenta filas y no aplica aún la adaptación para nulos. |

La versión propuesta con pesos se llama `info` y aparece en la siguiente celda.
No reemplaza automáticamente a `impurityEval_1`.


In [ ]:
from collections import Counter  # Usa Python estándar para contar las clases de D.
from math import log2  # Usa el logaritmo en base 2 para calcular entropía.


def impurityEval_1(D):  # L7: define impurityEval-1(D), la impureza antes de dividir.
    if not D:  # Caso auxiliar: comprueba si el conjunto de datos está vacío.
        return 0.0  # Asigna impureza cero al conjunto vacío.
    counts = Counter(row[-1] for row in D)  # Cuenta las clases de la última columna para obtener sus probabilidades.
    return -sum((count / len(D)) * log2(count / len(D))  # L7: calcula H(D) = -suma de p(c) por log2(p(c)).
                for count in counts.values())  # Incluye en la entropía cada clase presente en D.


def partition(Ai, D, threshold=None):  # L16: define cómo dividir D según los valores del atributo Ai.
    groups = {}  # Prepara los subconjuntos Dj, uno por rama.
    for row in D:  # Recorre las observaciones para asignarlas a su subconjunto.
        branch = row[Ai] if threshold is None else row[Ai] <= threshold  # L16: usa una categoría o, para datos numéricos, la comparación con el umbral.
        groups.setdefault(branch, []).append(row)  # L16: agrega la observación al subconjunto Dj que le corresponde.
    return groups  # Devuelve las particiones no vacías para evaluar o entrenar los hijos.


def impurityEval_2(Ai, D, threshold=None):  # L9: define impurityEval-2(Ai, D), la impureza después de dividir.
    if not D:  # Caso auxiliar: comprueba si el conjunto de datos está vacío.
        return 0.0  # Asigna impureza cero al conjunto vacío.
    groups = partition(Ai, D, threshold)  # L16: obtiene los subconjuntos necesarios para evaluar esta división.
    return sum(len(group) / len(D) * impurityEval_1(group)  # L9: suma la entropía de cada subconjunto ponderada por su tamaño.
               for group in groups.values())  # Considera todos los subconjuntos producidos por Ai.


## 4. Funciones aprobadas y fuentes

**Auxiliares aprobados; la integración está en la sección 6, no en el entrenador original.**
La siguiente celda sólo importa y define funciones: no reemplaza los auxiliares base
ni modifica `Node`. Cada función también incluye su fuente conceptual en la documentación.

### 4.1 Casos ponderados y valores desconocidos

Fuente principal: **NUL**, §§3.1–3.2, libro pp. 27–32 / PDF pp. 1–6.

| Función | Qué toma del texto | Páginas del libro / del PDF |
| --- | --- | --- |
| `es_desconocido` | Reconocer un atributo cuyo valor no se conoce; el texto usa `"?"`. | NUL 27–30 / 1–4 |
| `pesos_casos` | Asociar un peso a cada caso y propagar pesos no unitarios. | NUL 29 / 3 |
| `frecuencias_clase` | Reemplazar conteos enteros por sumas de pesos de casos fraccionarios. | NUL 29–31 / 3–5 |
| `probabilidades_clase` | Representar una clasificación mediante una distribución de clases. | NUL 30–32 / 4–6 |
| `info` | Calcular `info(T)` mediante probabilidades de clases. | NUL 28–30 / 2–4 |
| `separar_casos_conocidos` | Distinguir resultados conocidos y desconocidos de una prueba. | NUL 28–29 / 2–3 |
| `evaluar_prueba` | Calcular ganancia con la fracción conocida y una categoría adicional en la información de división. | NUL 28–31 / 2–5 |
| `particionar_con_desconocidos` | Repartir cada peso `w` entre ramas según sus probabilidades conocidas. | NUL 29–31 / 3–5 |

`info` conserva la notación del libro. Los demás nombres de esta tabla son descriptivos
propios. `info_X(T)`, `gain(X)` y `split info(X)` se calculan dentro de
`evaluar_prueba`; sus resultados se exponen mediante las claves `gain`, `split_info`
y `gain_ratio`, además de los pesos y probabilidades de las ramas.

**Decisiones de implementación:** los pesos se guardan separados de las filas;
se aceptan `None` y NaN además de `"?"`; se validan pesos finitos no negativos.
Las filas deben ser rectangulares y tener clase conocida en la última columna.
Los pesos cero no aportan a los cálculos. Estas convenciones Python no se transcriben del PDF.

La evaluación usa sólo casos conocidos y multiplica la ganancia por su fracción de peso.
`split_info` incluye el peso desconocido como grupo adicional, **no como rama del árbol**.
`branch_weights` guarda pesos conocidos y `branch_probabilities` los normaliza.
Una prueba sin dos ramas conocidas utilizables devuelve `None` por convención local.

La partición devuelve `(grupos, probabilidades)`; cada grupo contiene `(filas, pesos)`.
Las listas de salida son nuevas y las filas son referencias que no se modifican.
Si no hay peso conocido devuelve `({}, {})`: el entrenador enriquecido se detiene,
no inventar probabilidades. Esta respuesta es una decisión local.

### 4.2 Atributos continuos

| Función | Fuente y procedimiento | Páginas del libro / del PDF |
| --- | --- | --- |
| `evaluar_umbrales_continuos` | CON, §2.4: ordenar valores distintos, evaluar cortes y representar el umbral con un valor observado del entrenamiento. | 25–26 / 1–2 |

El PDF describe el procedimiento, pero no le asigna este nombre de función.
El atributo numérico lo elige explícitamente quien llama al auxiliar; no se convierten
categorías como `"hot"` en mediciones. Se devuelven diccionarios de candidatos,
no un árbol ni una elección automática entre ganancia y cociente de ganancia.

El mínimo predeterminado es un peso conocido de 2 por lado, basado en la restricción
de tamaño mínimo del texto (CON, libro p. 25 / PDF p. 1), adaptada a pesos.
Pasá los valores del atributo de **todo el entrenamiento** como `reference_values`.
Si no los pasás, se usan sólo las filas recibidas. El umbral observado no supera el
punto medio y queda por debajo del siguiente valor distinto; esta última protección
evita cortes inválidos por redondeo numérico.

La implementación evalúa directamente cada intervalo para facilitar su revisión.
La optimización en un único recorrido que menciona el texto no se implementa.
La sección 6 sí reutiliza el atributo continuo en la recursión.

**MDL opcional (`correccion_mdl=False` por defecto).** Cada candidato conserva
`gain`, `threshold`, `gain_ajustada`, `mdl_penalty` y `gain_ratio`. Se calcula:
`mdl_penalty = log2(K) / peso_total_del_nodo` y `gain_ajustada = gain - mdl_penalty`.
`gain_ratio = gain_ajustada / split_info` (cero si no hay información de división).
La entropía y `gain` originales no cambian. `gain` ya incluye la fracción conocida:
**no se vuelve a multiplicar la penalización por esa fracción**. El denominador
incluye todo el peso, también el desconocido y el recibido por reparto ancestral.

K cuenta fronteras entre valores distintos de peso positivo que satisfacen
`min_peso_rama`, `min_samples_leaf` y `min_weight_leaf`, antes de mirar la ganancia.
No se cuentan sólo cambios de clase. Con K=1 la penalización es cero; sin cortes
se devuelve `[]`, sin evaluar logaritmos. Primero se elige el umbral factible por
`gain` crudo; después se compara un representante por atributo usando su ajuste.

Fuente: [Quinlan (1996), pp. 79–80 y nota 2](https://www.cs.cmu.edu/afs/cs/project/jair/pub/volume4/quinlan96a.pdf).
El original usa `log2(N_distintos - 1) / |D|`; aquí se adopta la adaptación ponderada
de `Weka C45Split.handleNumericAttribute`: K sólo incluye fronteras admisibles y
el denominador es peso total. Los mínimos de hoja agregados también restringen K.
Con datos completos, pesos unitarios y todas las fronteras admisibles coinciden.
No es una reproducción de la enumeración completa original ni se penaliza dos veces.

### 4.3 Predicción con valores desconocidos

| Función | Fuente y procedimiento | Páginas del libro / del PDF |
| --- | --- | --- |
| `combinar_probabilidades_clase` | NUL, §§3.1.3–3.3: combinar las clasificaciones de varias ramas ponderadas por sus probabilidades. | 30–33 / 4–7 |

El nombre es propio. Quien llama debe proporcionar distribuciones de clases normalizadas
y probabilidades normalizadas con las mismas claves de ramas. No se conecta
con `predict_one`: el nuevo predictor enriquecido realiza el recorrido por varias ramas.

### 4.4 Poda basada en errores

| Función | Qué toma del texto | Páginas del libro / del PDF |
| --- | --- | --- |
| `U_CF` | POD, §§4.2–4.3: notación de la tasa superior estimada `U_CF(E,N)`. | 41–42 / 5–6 |
| `errores_estimados` | POD, §§4.2–4.3: conteo estimado `N * U_CF(E,N)`. | 40–42 / 4–6 |
| `estadisticas_hoja` | POD, §§4.2–4.3: clase mayoritaria, casos cubiertos y errores de una hoja hipotética. | 37–42 / 1–6 |
| `elegir_poda` | POD, §4.2: comparar el subárbol con una hoja y con la rama más utilizada. | 37–39 / 1–3 |

Sólo `U_CF` usa un nombre matemático explícito del libro; el resto son auxiliares propios.
`U_CF(E,N,CF)` devuelve una **tasa aproximada** y `errores_estimados(N,E,confidence)`
devuelve un **conteo aproximado**. Se mantiene `CF=0.25` como valor predeterminado.

**La rutina numérica no se transcribe del PDF:** el extracto describe el límite binomial,
pero no proporciona la tabla de coeficientes, la constante `0.67` ni las reglas de
interpolación usadas en esta propuesta. Se conserva la aproximación previa, no una
inversión binomial exacta. Para pesos totales menores que 1, el ancla de interpolación
es el propio peso total; esa extensión local fue aprobada. Para `N=0`,
devolver tasa cero es una convención propia, no una definición del libro.

`elegir_poda` devuelve `"keep"`, `"leaf"` o `"raise"`: conservar, sustituir por hoja
o elevar una rama. Son etiquetas propias, no comandos del PDF. El subárbol, la hoja y
la rama elevada deben evaluarse sobre **todos los casos que llegaban al nodo original**.
La sección 6 recalibra y estima la rama elevada sobre esa población completa.

Ante igualdad con el subárbol actual se conserva éste; entre mejoras iguales se prefiere
la hoja. La segunda regla es una decisión local. La sección 6 implementa el recorrido
de abajo hacia arriba, el reemplazo de nodos y la reevaluación de una rama elevada.




In [ ]:
from math import expm1, isfinite, isnan, log, sqrt


def es_desconocido(value):
    """Reconoce None, "?" y NaN como valores desconocidos.
    Fuente conceptual: NUL, libro pp. 27–30; PDF pp. 1–4, §§3.1–3.2.
    El libro usa "?"; admitir None y NaN es una decisión de implementación.
    El nombre del auxiliar no aparece como función en el PDF.
    """
    return (value is None or (isinstance(value, str) and value == "?")
            or (isinstance(value, float) and isnan(value)))


def pesos_casos(rows, weights=None):
    """Prepara pesos separados, finitos y no negativos; por defecto usa uno por caso.
    Fuente conceptual: NUL, libro p. 29; PDF p. 3, §3.1.2.
    El concepto de peso procede del texto; la validación y el nombre son propios.
    """
    result = [1.0] * len(rows) if weights is None else [float(w) for w in weights]
    if len(result) != len(rows) or any(not isfinite(w) or w < 0 for w in result):
        raise ValueError("Provide one finite nonnegative weight per row.")
    return result


def frecuencias_clase(rows, weights=None):
    """Suma los pesos por clase sin modificar las filas ni sus pesos.
    Fuente conceptual: NUL, libro pp. 29–31; PDF pp. 3–5, §§3.1.2–3.2.
    Generaliza los conteos a casos fraccionarios; el nombre del auxiliar es propio.
    """
    counts = {}
    for row, weight in zip(rows, pesos_casos(rows, weights)):
        if weight > 0:
            counts[row[-1]] = counts.get(row[-1], 0.0) + weight
    return counts


def probabilidades_clase(rows, weights=None):
    """Normaliza las frecuencias ponderadas de las clases.
    Fuente conceptual: NUL, libro pp. 30–32; PDF pp. 4–6, §§3.1.3–3.2.
    El nombre es propio; devolver un diccionario vacío si no hay peso es una convención.
    """
    counts = frecuencias_clase(rows, weights)
    total = sum(counts.values())
    return {label: count / total for label, count in counts.items()}


def info(rows, weights=None):
    """Calcula info(T), la entropía de clases, admitiendo pesos por caso.
    Fuente conceptual: NUL, libro pp. 28–30; PDF pp. 2–4, §§3.1.1–3.2.
    info es la notación del libro; el argumento opcional de pesos es su adaptación a Python.
    """
    return -sum(p * log2(p) for p in probabilidades_clase(rows, weights).values())


def separar_casos_conocidos(rows, attribute, weights=None, threshold=None):
    """Separa las ramas conocidas de los casos con resultado desconocido.
    Fuente conceptual: NUL, libro pp. 28–29; PDF pp. 2–3, §§3.1.1–3.1.2.
    Devuelve grupos de filas/pesos y pares desconocidos; el nombre y formato son propios.
    """
    groups, unknown = {}, []
    for row, weight in zip(rows, pesos_casos(rows, weights)):
        if weight == 0:
            continue
        value = row[attribute]
        if es_desconocido(value):
            unknown.append((row, weight))
        else:
            branch = value if threshold is None else value <= threshold
            group, group_weights = groups.setdefault(branch, ([], []))
            group.append(row)
            group_weights.append(weight)
    return groups, unknown


def evaluar_prueba(rows, attribute, weights=None, threshold=None):
    """Calcula gain(X), split info(X) y su cociente para una prueba.
    Fuente conceptual: NUL, libro pp. 28–31; PDF pp. 2–5, §§3.1.1–3.2.
    El nombre agregado es propio: las expresiones del libro son campos del resultado.
    Una prueba sin dos ramas conocidas utilizables devuelve None por convención local.
    """
    groups, unknown = separar_casos_conocidos(rows, attribute, weights, threshold)
    masses = {branch: sum(w) for branch, (_, w) in groups.items()}
    known_weight = sum(masses.values())
    total = known_weight + sum(w for _, w in unknown)
    if len(groups) < 2 or known_weight == 0:
        return None
    known_rows = [row for group, _ in groups.values() for row in group]
    known_weights = [w for _, group_weights in groups.values() for w in group_weights]
    after = sum(masses[branch] / known_weight * info(group, group_weights)
                for branch, (group, group_weights) in groups.items())
    gain = known_weight / total * (info(known_rows, known_weights) - after)
    proportions = [mass / total for mass in masses.values()]
    if total > known_weight:
        proportions.append((total - known_weight) / total)
    split_info = -sum(p * log2(p) for p in proportions)
    return {"threshold": threshold, "gain": gain, "split_info": split_info,
            "gain_ratio": gain / split_info if split_info > 0 else 0.0, "branch_weights": masses,
            "branch_probabilities": {branch: mass / known_weight
                                     for branch, mass in masses.items()}}


def particionar_con_desconocidos(rows, attribute, weights=None, threshold=None):
    """Reparte cada caso desconocido con peso w por la probabilidad de cada rama.
    Fuente conceptual: NUL, libro pp. 29–31; PDF pp. 3–5, §§3.1.2–3.2.
    No crea una rama para nulos. El nombre y el formato (grupos, probabilidades) son propios.
    """
    groups, unknown = separar_casos_conocidos(rows, attribute, weights, threshold)
    known_weight = sum(sum(w) for _, w in groups.values())
    if known_weight == 0:
        return {}, {}
    probabilities = {branch: sum(w) / known_weight for branch, (_, w) in groups.items()}
    for branch, (group, group_weights) in groups.items():
        for row, weight in unknown:
            group.append(row)
            group_weights.append(weight * probabilities[branch])
    return groups, probabilities


def _validar_minimos_hoja(min_samples_leaf, min_weight_leaf):
    """Valida restricciones propias de factibilidad, no reglas de poda.
    Fuente conceptual: NUL, libro pp. 29–31; PDF pp. 3–5, pesos frente a conteos.
    Los mínimos configurables son extensiones de esta implementación.
    """
    if (isinstance(min_samples_leaf, bool) or not isinstance(min_samples_leaf, int)
            or min_samples_leaf < 0):
        raise ValueError('min_samples_leaf debe ser un entero no negativo.')
    if not isfinite(min_weight_leaf) or min_weight_leaf < 0:
        raise ValueError('min_weight_leaf debe ser finito y no negativo.')


def _cumple_minimos_hoja(filas, atributo, pesos, prueba, originales,
                         min_samples_leaf, min_weight_leaf):
    """Cuenta originales conocidos; exige masa total tras repartir desconocidos.
    Fuente conceptual: NUL, libro pp. 29–31; PDF pp. 3–5, reparto fraccionario.
    La elegibilidad ancestral y los mínimos de hoja son convenciones propias.
    """
    conteos = dict.fromkeys(prueba['branch_weights'], 0)
    for fila, peso, original in zip(filas, pesos, originales):
        valor = fila[atributo]
        if original and peso > 0 and not es_desconocido(valor):
            rama = valor if prueba['threshold'] is None else valor <= prueba['threshold']
            conteos[rama] += 1
    grupos, _ = particionar_con_desconocidos(filas, atributo, pesos, prueba['threshold'])
    return all(conteos[rama] >= min_samples_leaf and sum(ponderaciones) >= min_weight_leaf
               for rama, (_, ponderaciones) in grupos.items())


def evaluar_umbrales_continuos(rows, attribute, weights=None, reference_values=None,
                      min_branch_weight=2.0, *, correccion_mdl=False,
                      min_samples_leaf=0, min_weight_leaf=0.0, originales=None):
    """Devuelve las estadísticas de cortes binarios válidos, ordenados por umbral.
    Fuente conceptual: CON, libro pp. 25–26; PDF pp. 1–2, §2.4.
    El texto describe el procedimiento, no este nombre de función.
    Usa valores observados de referencia y peso conocido mínimo; no elige ni entrena.
    MDL: Quinlan (1996), pp. 79–80; adaptación ponderada de Weka C45Split.
    K cuenta todas las fronteras factibles, sin depender de cambios de clase.
    Resta log2(K)/peso_total una sola vez al gain ya ajustado por desconocidos.
    originales marca casos sin reparto ancestral; None los considera originales.
    """
    _validar_minimos_hoja(min_samples_leaf, min_weight_leaf)
    if not isinstance(correccion_mdl, bool):
        raise ValueError('correccion_mdl debe ser booleana.')
    if not isfinite(min_branch_weight) or min_branch_weight <= 0:
        raise ValueError("min_branch_weight must be finite and positive.")
    weights = pesos_casos(rows, weights)
    originales = [w > 0 for w in weights] if originales is None else originales
    values = {row[attribute] for row, w in zip(rows, weights)
              if w > 0 and not es_desconocido(row[attribute])}
    reference = values if reference_values is None else {
        value for value in reference_values if not es_desconocido(value)}
    if any(not isinstance(v, (int, float)) or not isfinite(v) for v in values | reference):
        raise ValueError("Continuous values must be finite numbers.")
    if not values <= reference:
        raise ValueError("reference_values must include every known positive-weight value.")
    ordered = sorted(values)
    candidates = []
    for left, right in zip(ordered, ordered[1:]):
        midpoint = left / 2 + right / 2
        threshold = max((v for v in reference if v < right and v <= midpoint), default=left)
        stats = evaluar_prueba(rows, attribute, weights, threshold)
        if (stats is not None and min(stats['branch_weights'].values()) >= min_branch_weight
                and _cumple_minimos_hoja(rows, attribute, weights, stats, originales,
                                         min_samples_leaf, min_weight_leaf)):
            candidates.append(stats)
    if not candidates:
        return []
    penalty = log2(len(candidates)) / sum(weights) if correccion_mdl else 0.0
    for stats in candidates:
        stats['mdl_penalty'] = penalty
        stats['gain_ajustada'] = stats['gain'] - penalty
        stats['gain_ratio'] = (stats['gain_ajustada'] / stats['split_info']
                               if stats['split_info'] > 0 else 0.0)
    return candidates


def combinar_probabilidades_clase(distributions, branch_probabilities):
    """Combina distribuciones de clases con las probabilidades de sus ramas.
    Fuente conceptual: NUL, libro pp. 30–33; PDF pp. 4–7, §§3.1.3–3.3.
    El nombre es propio. Requiere distribuciones normalizadas y claves de ramas coincidentes.
    """
    combined = {}
    for branch, branch_probability in branch_probabilities.items():
        for label, probability in distributions[branch].items():
            combined[label] = combined.get(label, 0.0) + branch_probability * probability
    return combined


def errores_estimados(total_weight, observed_errors, confidence=0.25):
    """Aproxima el conteo de errores N por U_CF(E,N), no la tasa U_CF sola.
    Fuente conceptual: POD, libro pp. 40–42; PDF pp. 4–6, §§4.2–4.3.
    El nombre, tabla de coeficientes y rutina numérica no se transcriben del extracto.
    La aproximación existente interpola errores menores que uno entre cero y min(1,N).
    El ancla N para hojas de peso menor que uno es una extensión propia pendiente de aprobación.
    """
    n, e = total_weight, observed_errors
    if (not all(isfinite(v) for v in (n, e, confidence))
            or not 0 <= e <= n or not 0 < confidence < 1):
        raise ValueError("Require 0 <= errors <= weight and 0 < confidence < 1.")
    if n == 0:
        return 0.0
    zero_errors = -n * expm1(log(confidence) / n)
    if e == 0:
        return zero_errors
    if e == n:
        return n
    if e < 1:
        anchor = min(1.0, n)
        upper = errores_estimados(n, anchor, confidence)
        return zero_errors + e / anchor * (upper - zero_errors)
    if e + 0.5 >= n:
        return e + 0.67 * (n - e)
    levels = (0.0, 0.001, 0.005, 0.01, 0.05, 0.10, 0.20, 0.40, 1.0)
    deviations = (4.0, 3.09, 2.58, 2.33, 1.65, 1.28, 0.84, 0.25, 0.0)
    index = next(i for i in range(1, len(levels)) if confidence <= levels[i])
    fraction = (confidence - levels[index - 1]) / (levels[index] - levels[index - 1])
    z = deviations[index - 1] + fraction * (deviations[index] - deviations[index - 1])
    f = (e + 0.5) / n
    upper = (f + z * z / (2 * n) + z * sqrt(f * (1 - f) / n + z * z / (4 * n * n)))
    return n * upper / (1 + z * z / n)


def U_CF(E, N, CF=0.25):
    """Devuelve una aproximación a la tasa U_CF(E,N), no al número de errores.
    Fuente conceptual: POD, libro pp. 41–42; PDF pp. 5–6, §§4.2–4.3.
    Conserva la notación y el orden E,N del libro; usa la rutina aproximada propuesta.
    Para N=0 devuelve cero por convención local, no por una definición del PDF.
    """
    estimated = errores_estimados(N, E, CF)
    return estimated / N if N else 0.0


def estadisticas_hoja(rows, weights=None, confidence=0.25):
    """Describe una hoja mayoritaria hipotética, sin crear ni modificar nodos.
    Fuente conceptual: POD, libro pp. 37–42; PDF pp. 1–6, §§4.2–4.3.
    El nombre y el diccionario son propios; usa frecuencias y errores ponderados.
    """
    counts = frecuencias_clase(rows, weights)
    total = sum(counts.values())
    prediction = max(counts, key=counts.get, default=None)
    errors = total - max(counts.values(), default=0.0)
    return {"prediction": prediction, "class_counts": counts, "total_weight": total,
            "observed_errors": errors,
            "estimated_errors": errores_estimados(total, errors, confidence)}


def elegir_poda(subtree_errors, leaf_errors, raised_branch_errors=None):
    """Compara errores del subárbol, la hoja y una rama elevada sobre los mismos casos.
    Fuente conceptual: POD, libro pp. 37–39; PDF pp. 1–3, §4.2.
    El nombre y las etiquetas keep/leaf/raise son propios. No recorre ni modifica el árbol.
    Ante igualdad con el subárbol actual lo conserva; entre mejoras iguales prefiere la hoja.
    """
    candidates = [("keep", subtree_errors), ("leaf", leaf_errors)]
    if raised_branch_errors is not None:
        candidates.append(("raise", raised_branch_errors))
    return min(candidates, key=lambda candidate: candidate[1])[0]


## 6. Entrenamiento enriquecido — integración aprobada

**Usá `entrenar_arbol_enriquecido(D, A)` para la nueva implementación.**
La demostración guarda `arbol_enriquecido` sin imprimir ni sobrescribir `tree`.
`decision_tree`, `predict_one`, `print_tree`, `Node`, datos y auxiliares siguen intactos.

```python
arbol = entrenar_arbol_enriquecido(D, A, continuos=(), pesos=None,
    min_gain=1e-12, min_peso_rama=2.0, confianza=0.25, podar=True)
distribucion = predecir_probabilidades_enriquecido(arbol, [None, 'cool', 'normal', 'weak'])
clase = predecir_enriquecido(arbol, [None, 'cool', 'normal', 'weak'])
sin_poda = entrenar_arbol_enriquecido(D, A, podar=False)
```

| Mecanismo integrado | Contrato y fuente conceptual |
| --- | --- |
| Selección ponderada | `criterio='gain'` por defecto; admite `'gain_ratio'`. `min_gain` exige ganancia cruda escalada por peso nodo/raíz, nunca ratio ni ajuste MDL. Empates por orden de atributos y luego umbral ascendente. NUL, libro pp. 28–31 / PDF pp. 2–5. |
| Continuos explícitos | `continuos` contiene índices, nunca se infieren por ser números. Cortes binarios `<=`/`>`, referencia de todo el entrenamiento y mínimo de peso conocido por lado `min_peso_rama` (sólo cortes continuos). Se reutilizan continuos y se retiran categóricos. CON, libro pp. 25–26 / PDF pp. 1–2. |
| Nulos y estado | Reparto fraccionario, frecuencias, peso total y probabilidades en `NodoEnriquecido`, separado de `Node` y compatible con `print_tree`. Sin peso conocido se devuelve hoja mayoritaria. NUL, libro pp. 29–32 / PDF pp. 3–6. |
| Predicción | Nulos recorren todas las ramas y combinan distribuciones normalizadas. Una categoría nueva devuelve la distribución del nodo; la clase empata por primer orden de clase del nodo. Este respaldo determinista es propio. NUL, libro pp. 30–33 / PDF pp. 4–7. |
| Poda pesimista (C4.5) | Recorrido ascendente: conservar, reemplazar por hoja o elevar la rama de mayor peso (primer orden de rama ante empate). Todas las alternativas se evalúan sobre la población completa del nodo. POD, libro pp. 37–43 / PDF pp. 1–7. |

La rama elevada se copia, se redirigen todos los casos del padre y se recalculan
distribuciones, pesos y probabilidades, sin reutilizar errores de su antigua población.
Los nulos se reparten por las proporciones conocidas de esa nueva población.
Las categorías que esa estructura no reconoce quedan en el respaldo del nodo: su
contribución al error se estima separadamente con la predicción del nodo. Sin ramas
reconocidas con peso conocido la estructura se convierte en hoja. No se inventan proporciones.
Los errores del subárbol son la suma de estimaciones de sus hojas y respaldos.

**Adaptación, no transcripción:** las APIs, estructuras y recalibración son código Python
propio. La poda conserva exactamente la aproximación aprobada `errores_estimados` / `U_CF`,
incluida la extensión fraccionaria; no es inversión binomial exacta ni rutina literal del libro.
Las filas deben tener clase conocida al final. Los pesos van separados y no se modifican
filas, listas de atributos ni pesos del llamador. Sin peso positivo se obtiene una hoja
sin clase (`None`) y una distribución vacía; no hay clasificación normalizable en ese caso.
Esta implementación didáctica evalúa intervalos directamente y recalibra copias; no
implementa la optimización de barrido único ni pretende ser un C4.5 completo.

### Selección y mínimos de hoja

Los seis parámetros nuevos son **sólo por nombre**, después de los diez argumentos
posicionales existentes; las llamadas anteriores conservan sus valores predeterminados.

| Parámetro | Predeterminado y efecto |
| --- | --- |
| `criterio` | `'gain'`: compara ganancia ajustada; `'gain_ratio'`: compara su cociente. |
| `filtro_ganancia_promedio` | `True`, sólo para ratio: exige ganancia ajustada al menos igual al promedio. |
| `min_gain_ratio` | `None`: sin mínimo; si se proporciona, acepta la prueba elegida sólo si su ratio alcanza ese valor, incluso en modo gain. |
| `correccion_mdl` | `False`: no penaliza; `True`: ajuste descrito en §4.2 y ganancia ajustada final estrictamente positiva. |
| `min_samples_leaf` | `0` desactiva el mínimo; entero no negativo de originales conocidos por rama. |
| `min_weight_leaf` | `0.0`: mínimo finito no negativo de masa total por rama tras el reparto. |

La estrategia ratio de C4.5 promedia **un representante por atributo factible**,
incluidos los de ganancia cero (o ajustada negativa con MDL); constantes y atributos
sin dos ramas conocidas no participan. No se agrega la heurística de cardinalidad
de Weka. La ganancia estrictamente positiva se exige al seleccionar, no al promediar.
El filtro promedio no afecta al modo gain. `min_gain_ratio` se comprueba después de
elegir, sin buscar un segundo atributo si falla; la igualdad con el mínimo se acepta.
En ambos modos `min_gain` mantiene `(peso_nodo / peso_raíz) * gain >= min_gain`.

**Observación no equivale a peso.** `min_samples_leaf` cuenta sólo filas de peso
positivo conocidas en esta prueba y que nunca fueron repartidas por un desconocido
ancestral. Una bandera booleana acompaña cada fila: una vez fraccionada, nunca
recupera elegibilidad, aunque otro atributo sea conocido. El peso puede ser menor
que uno sin perder la condición de original si no hubo reparto. Los desconocidos
no suman observaciones, pero toda su masa sí cuenta para `min_weight_leaf`.

Ambos mínimos se comprueban antes de ordenar candidatos categóricos o continuos;
un corte inviable no oculta otro corte o atributo viable. `min_samples_split` sigue
contando `len(filas)`, independientemente del peso y de estas banderas.
`min_peso_rama` sólo exige peso **conocido** por lado continuo; `min_weight_leaf`
exige peso **total** posterior al reparto en cualquier tipo de atributo. Son
restricciones de crecimiento, no poda. El ratio, el reparto y MDL proceden de C4.5;
la semántica configurable de mínimos es una extensión local, no una selección CART.

### 6.1 Soporte de integración

La siguiente celda reúne el estado ponderado, la selección de pruebas, la redirección
de poblaciones, la poda y los predictores. Sólo define auxiliares; no entrena ni imprime.
Son adaptadores propios de las funciones aprobadas en la sección 4.
Separar este soporte permite leer después el crecimiento recursivo como el algoritmo base.

### Métodos de poda: elegir sin cambiar el crecimiento

Los nuevos argumentos son **exclusivamente por nombre**, después de los diez existentes:
`metodo_poda=None`, `subtree_raising=True`, `ccp_alpha=0.0`,
`filas_validacion=None` y `pesos_validacion=None`. La poda ocurre después de crecer.

| `podar` | `metodo_poda` | Método efectivo |
| --- | --- | --- |
| `True` | `None` | `pesimista` (comportamiento anterior) |
| `False` | `None` o `ninguna` | `ninguna` |
| `True` | Cualquiera de los cuatro nombres explícitos | El solicitado, incluso `ninguna` |
| `False` | `pesimista`, `reduced_error` o `cost_complexity` | `ValueError`: combinación contradictoria |

- **`pesimista`:** conserva el estimador aprobado, `confianza` y los desempates anteriores.
  `subtree_raising=False` omite sólo la elevación en todos los niveles; sigue permitiendo
  el reemplazo por hoja. Este control sólo afecta a este método.
- **`reduced_error`:** exige validación explícita no vacía, del mismo ancho y con clase
  al final. `pesos_validacion` pasa por `pesos_casos`; su total debe ser positivo y finito.
  Sin pesos, cada fila pesa uno. No usa el estimador pesimista ni `confianza`.
- **`cost_complexity`:** usa el entrenamiento para minimizar
  `R_alpha(T) = sum(w_i * [predecir_enriquecido(T, x_i) != y_i]) + ccp_alpha * hojas(T)`.
  El error es un **conteo ponderado crudo**, no una tasa: escalar pesos cambia la escala
  apropiada de alpha. `ccp_alpha` debe ser numérico, finito y no negativo.
- **`ninguna`:** conserva el árbol crecido; sólo agrega el resumen estructural.

**Un mismo riesgo 0-1 para las dos alternativas:** reduced-error usa la misma suma
sobre validación, sin penalización. En ambos métodos se llama al predictor público:
primero combina probabilidades ante nulos y luego hace argmax determinista por orden
de clase de la raíz. No se usa `1-p(clase verdadera)` ni suma de errores de hojas.
Las categorías nuevas mantienen el respaldo del nodo. Las hojas candidatas conservan
las frecuencias, distribución y mayoría del entrenamiento; validación nunca las recalibra.

**Algoritmo greedy, no programación dinámica independiente:** una sola pasada ascendente
en orden de inserción de ramas compara el árbol completo actual contra el mismo árbol
con ese nodo convertido en hoja. Cada comparación usa toda la población relevante
(validación o entrenamiento), con las podas ya aceptadas. A igualdad de costo se poda.
Con nulos, una mejora fraccionaria local puede empeorar el argmax de una mezcla en un
ancestro; por eso no se suman óptimos independientes por hijo. No garantiza el mínimo
global entre todas las combinaciones posibles. Se trabaja sobre copias, sin modificar
el árbol crecido ni los datos. Cost-complexity es una extensión **inspirada en CART**,
con alpha fijo, no el procedimiento CART completo de weakest-link ni una ruta de alphas.
Reduced-error también es una alternativa; no se presenta como poda pesimista de C4.5.

**Sin partición implícita ni fuga de datos:** sólo reduced-error admite argumentos de
validación; los otros métodos los rechazan. Los errores de configuración se detectan
antes de crecer. Se rechaza `filas_validacion is filas`, pero no se puede detectar
solapamiento semántico ni saber si una copia proviene del test final. Es obligación del
llamador proporcionar un conjunto independiente y nunca usar el test final para podar.

```python
arbol = entrenar_arbol_enriquecido(D, A, metodo_poda='reduced_error',
    filas_validacion=validacion_independiente, pesos_validacion=pesos_independientes)
arbol_cart = entrenar_arbol_enriquecido(D, A, metodo_poda='cost_complexity', ccp_alpha=0.5)
```

`estadisticas_arbol_enriquecido(arbol)` devuelve `{'profundidad': ..., 'nodos': ...,
'hojas': ...}`; profundidad cuenta aristas desde la raíz en cero. Cada raíz entrenada
agrega `resumen_poda = {'antes': estadisticas, 'despues': estadisticas, 'metodo': nombre}`
con el método efectivo. El panel puede mostrar la comparación sin reentrenar. Estos
metadatos no cambian el estado previo ni las predicciones de `NodoEnriquecido`.


In [ ]:
class NodoEnriquecido:
    """Estado ponderado propio; NUL, libro pp. 29–32 / PDF pp. 3–6."""
    def __init__(self, filas, pesos):
        self.class_counts = frecuencias_clase(filas, pesos)
        self.total_weight = sum(self.class_counts.values())
        self.class_probabilities = probabilidades_clase(filas, pesos)
        self.prediction = max(self.class_counts, key=self.class_counts.get, default=None)
        self.attribute = None
        self.threshold = None
        self.children = {}
        self.branch_probabilities = {}


def _seleccionar_prueba_enriquecida(filas, atributos, pesos, continuos, referencias, minimo, *,
                                    criterio='gain', filtro_ganancia_promedio=True,
                                    min_gain_ratio=None, correccion_mdl=False,
                                    min_samples_leaf=0, min_weight_leaf=0.0, originales=None):
    """Compara un representante factible por atributo; conserva el primer empate.
    NUL, libro pp. 28–31 / PDF pp. 2–5; CON, libro pp. 25–26 / PDF pp. 1–2.
    El umbral representa al atributo por gain crudo, no por ratio ni MDL.
    La comparación y el promedio usan gain ajustada; min_gain se comprueba al crecer.
    min_gain_ratio es aceptación posterior, también en modo gain (sin reelegir).
    """
    originales = [p > 0 for p in pesos] if originales is None else originales
    elegibles = []
    for atributo in atributos:
        if atributo in continuos:
            candidatos = evaluar_umbrales_continuos(
                filas, atributo, pesos, referencias[atributo], minimo,
                correccion_mdl=correccion_mdl, min_samples_leaf=min_samples_leaf,
                min_weight_leaf=min_weight_leaf, originales=originales)
        else:
            prueba = evaluar_prueba(filas, atributo, pesos)
            candidatos = []
            if prueba is not None and _cumple_minimos_hoja(
                    filas, atributo, pesos, prueba, originales, min_samples_leaf, min_weight_leaf):
                prueba.update(gain_ajustada=prueba['gain'], mdl_penalty=0.0)
                candidatos.append(prueba)
        if candidatos:
            prueba = max(candidatos, key=lambda c: c['gain'])
            elegibles.append((atributo, prueba))  # El promedio incluye ganancia cero.
    if not elegibles:
        return None
    if criterio == 'gain_ratio' and filtro_ganancia_promedio:
        promedio = sum(p['gain_ajustada'] for _, p in elegibles) / len(elegibles)
        elegibles = [(a, p) for a, p in elegibles if p['gain_ajustada'] >= promedio]
    elegibles = [(a, p) for a, p in elegibles if p['gain'] > 0
                 and (not correccion_mdl or p['gain_ajustada'] > 0)]
    if not elegibles:
        return None
    campo = 'gain_ratio' if criterio == 'gain_ratio' else 'gain_ajustada'
    mejor = max(elegibles, key=lambda candidato: candidato[1][campo])
    if min_gain_ratio is not None and mejor[1]['gain_ratio'] < min_gain_ratio:
        return None
    return mejor


def _originales_rama_enriquecida(filas, pesos, originales, atributo, umbral, rama):
    """Alinea elegibilidad con particionar_con_desconocidos: conocidos y luego nulos.
    Un caso repartido aquí o en un ancestro nunca vuelve a contar como original.
    """
    conocidos = [original for fila, peso, original in zip(filas, pesos, originales)
                 if peso > 0 and not es_desconocido(fila[atributo])
                 and (fila[atributo] if umbral is None else fila[atributo] <= umbral) == rama]
    repartidos = [False for fila, peso in zip(filas, pesos)
                  if peso > 0 and es_desconocido(fila[atributo])]
    return conocidos + repartidos


def _poblaciones_enriquecidas(nodo, filas, pesos):
    """Redirige la población completa; NUL pp. 29–32 / PDF 3–6; POD pp. 37–39 / PDF 1–3.
    Las categorías nuevas quedan en el respaldo; los nulos se reparten sin inventar ramas.
    """
    utilizables, ponderaciones, respaldo, pesos_respaldo = [], [], [], []
    for fila, peso in zip(filas, pesos):
        valor = fila[nodo.attribute]
        desconocido = es_desconocido(valor)
        rama = None if desconocido else (valor if nodo.threshold is None else valor <= nodo.threshold)
        if desconocido or rama in nodo.children:
            utilizables.append(fila)
            ponderaciones.append(peso)
        else:
            respaldo.append(fila)
            pesos_respaldo.append(peso)
    grupos, probabilidades = particionar_con_desconocidos(
        utilizables, nodo.attribute, ponderaciones, nodo.threshold)
    return grupos, probabilidades, respaldo, pesos_respaldo


def _error_respaldo_enriquecido(nodo, filas, pesos, confianza):
    """Estima el respaldo propio con la misma aproximación; POD pp. 40–42 / PDF 4–6."""
    errores = sum(p for f, p in zip(filas, pesos) if f[-1] != nodo.prediction)
    return errores_estimados(sum(pesos), errores, confianza)


def _recalibrar_enriquecido(plantilla, filas, pesos, confianza):
    """Copia y reestima sobre TODOS los casos; POD, libro pp. 37–43 / PDF pp. 1–7.
    No conserva estadísticas del antiguo subconjunto de una rama elevada.
    """
    nodo = NodoEnriquecido(filas, pesos)
    hoja = estadisticas_hoja(filas, pesos, confianza)
    if plantilla.attribute is None or nodo.total_weight == 0:
        return nodo, hoja['estimated_errors']
    grupos, probabilidades, respaldo, pesos_respaldo = _poblaciones_enriquecidas(plantilla, filas, pesos)
    if not grupos:
        return nodo, hoja['estimated_errors']
    nodo.attribute, nodo.threshold = plantilla.attribute, plantilla.threshold
    nodo.branch_probabilities = probabilidades
    error = _error_respaldo_enriquecido(nodo, respaldo, pesos_respaldo, confianza)
    for rama, (grupo, ponderaciones) in grupos.items():
        hijo, estimacion = _recalibrar_enriquecido(
            plantilla.children[rama], grupo, ponderaciones, confianza)
        nodo.children[rama] = hijo
        error += estimacion
    return nodo, error


def _podar_enriquecido(plantilla, filas, pesos, confianza, subtree_raising=True):
    """Poda ascendente real: hoja, conservación o elevación; POD pp. 37–43 / PDF 1–7.
    Compara las tres opciones en la misma población, con la aproximación aprobada.
    """
    nodo, error = _recalibrar_enriquecido(plantilla, filas, pesos, confianza)
    if nodo.attribute is None:
        return nodo, error
    grupos, _, respaldo, pesos_respaldo = _poblaciones_enriquecidas(nodo, filas, pesos)
    error = _error_respaldo_enriquecido(nodo, respaldo, pesos_respaldo, confianza)
    for rama, (grupo, ponderaciones) in grupos.items():
        nodo.children[rama], estimacion = _podar_enriquecido(
            nodo.children[rama], grupo, ponderaciones, confianza, subtree_raising)
        error += estimacion
    hoja = estadisticas_hoja(filas, pesos, confianza)
    elevada, error_elevada = None, None
    if subtree_raising:
        mas_usada = max(nodo.children.values(), key=lambda hijo: hijo.total_weight)
        elevada, error_elevada = _recalibrar_enriquecido(mas_usada, filas, pesos, confianza)
    decision = elegir_poda(error, hoja['estimated_errors'], error_elevada)
    if decision == 'leaf':
        return NodoEnriquecido(filas, pesos), hoja['estimated_errors']
    if decision == 'raise':
        return elevada, error_elevada
    return nodo, error


def estadisticas_arbol_enriquecido(arbol):
    """Describe estructura sin entrenar: profundidad desde raíz cero, nodos y hojas."""
    if arbol.attribute is None:
        return {'profundidad': 0, 'nodos': 1, 'hojas': 1}
    hijos = [estadisticas_arbol_enriquecido(hijo) for hijo in arbol.children.values()]
    return {'profundidad': 1 + max(h['profundidad'] for h in hijos),
            'nodos': 1 + sum(h['nodos'] for h in hijos),
            'hojas': sum(h['hojas'] for h in hijos)}


def _riesgo_enriquecido(arbol, filas, pesos):
    """Conteo ponderado 0-1 con el predictor público, no suma de errores de hojas."""
    return sum(peso for fila, peso in zip(filas, pesos)
               if predecir_enriquecido(arbol, fila) != fila[-1])


def _podar_por_riesgo_enriquecido(plantilla, filas, pesos, alpha=0.0):
    """Una pasada greedy ascendente; compara cada cambio sobre el árbol COMPLETO.
    Reduced-error usa validación y alpha=0; cost-complexity usa entrenamiento.
    Las mezclas por nulos impiden asumir riesgos aditivos por hijo. No es DP exacta
    ni weakest-link CART. Empates favorecen hoja; conserva estadísticas de entrenamiento.
    """
    from copy import deepcopy
    arbol = deepcopy(plantilla)

    def costo():
        return (_riesgo_enriquecido(arbol, filas, pesos)
                + alpha * estadisticas_arbol_enriquecido(arbol)['hojas'])

    def recorrer(nodo):
        if nodo.attribute is None:
            return
        for hijo in nodo.children.values():
            recorrer(hijo)
        anterior = costo()
        estado = nodo.attribute, nodo.threshold, nodo.children, nodo.branch_probabilities
        nodo.attribute, nodo.threshold, nodo.children, nodo.branch_probabilities = None, None, {}, {}
        if costo() > anterior:
            nodo.attribute, nodo.threshold, nodo.children, nodo.branch_probabilities = estado

    recorrer(arbol)
    return arbol


def _validar_poda_enriquecida(filas, podar, metodo, subtree_raising, alpha, validacion, pesos):
    """Resuelve compatibilidad y exige validación explícita antes de crecer."""
    if not isinstance(podar, bool) or not isinstance(subtree_raising, bool):
        raise ValueError('podar y subtree_raising deben ser booleanos.')
    if metodo is None:
        metodo = 'pesimista' if podar else 'ninguna'
    if metodo not in ('pesimista', 'reduced_error', 'cost_complexity', 'ninguna'):
        raise ValueError('metodo_poda debe ser pesimista, reduced_error, cost_complexity o ninguna.')
    if not podar and metodo != 'ninguna':
        raise ValueError('podar=False sólo admite metodo_poda=None o ninguna.')
    if (isinstance(alpha, bool) or not isinstance(alpha, (int, float))
            or not isfinite(alpha) or alpha < 0):
        raise ValueError('ccp_alpha debe ser finito y no negativo.')
    if metodo != 'reduced_error':
        if validacion is not None or pesos is not None:
            raise ValueError('filas_validacion y pesos_validacion sólo se admiten con reduced_error.')
        return metodo, None
    if validacion is filas:
        raise ValueError('La validación debe ser independiente del entrenamiento.')
    if validacion is None or len(validacion) == 0:
        raise ValueError('reduced_error requiere filas_validacion explícitas y no vacías.')
    if (not filas or any(len(fila) != len(filas[0]) for fila in validacion)
            or len(filas[0]) == 0):
        raise ValueError('La validación debe tener el mismo ancho que el entrenamiento, con clase al final.')
    ponderaciones = pesos_casos(validacion, pesos)
    if not isfinite(sum(ponderaciones)) or sum(ponderaciones) <= 0:
        raise ValueError('La validación debe tener peso total positivo y finito.')
    return metodo, ponderaciones


def predecir_probabilidades_enriquecido(arbol, fila):
    """Combina ramas ante nulos; NUL, libro pp. 30–33 / PDF pp. 4–7.
    Una categoría nueva usa la distribución del nodo como respaldo determinista propio.
    """
    if arbol.attribute is None:
        return dict(arbol.class_probabilities)
    valor = fila[arbol.attribute]
    if es_desconocido(valor):
        distribuciones = {rama: predecir_probabilidades_enriquecido(hijo, fila)
                          for rama, hijo in arbol.children.items()}
        combinada = combinar_probabilidades_clase(distribuciones, arbol.branch_probabilities)
        total = sum(combinada.values())
        return {clase: p / total for clase, p in combinada.items()} if total else {}
    rama = valor if arbol.threshold is None else valor <= arbol.threshold
    if rama not in arbol.children:
        return dict(arbol.class_probabilities)
    return predecir_probabilidades_enriquecido(arbol.children[rama], fila)


def predecir_enriquecido(arbol, fila):
    """Elige la clase más probable; empate por orden del nodo, convención propia.
    Fuente conceptual de la distribución: NUL, libro pp. 30–33 / PDF pp. 4–7.
    """
    probabilidades = predecir_probabilidades_enriquecido(arbol, fila)
    return max(arbol.class_counts, key=lambda clase: probabilidades.get(clase, 0), default=None)



### 6.2 Crecimiento comparable con el algoritmo base

El núcleo conserva la recursión directa y el orden de pasos de `decision_tree`.
`filas` y `atributos` corresponden a D y A; `Ag`, `group` y `remaining` hacen visible
la misma elección, partición y llamada recursiva. Los nuevos controles sólo afectan
al núcleo enriquecido; el algoritmo original permanece sin cambios.

| Paso base | Paso enriquecido y auxiliar |
| --- | --- |
| L1–L2: clase pura | Hoja si no hay peso positivo o hay una sola clase ponderada; `NodoEnriquecido` usa `frecuencias_clase` y `probabilidades_clase`. |
| L3–L4: sin atributos | Devuelve la hoja mayoritaria ya preparada; `info` conserva además la parada por entropía cero. |
| L7–L11: evaluar y elegir Ag | `_seleccionar_prueba_enriquecida` llama a `evaluar_prueba` o `evaluar_umbrales_continuos` según el índice explícito. |
| Controles de crecimiento | Devuelve hoja al alcanzar `max_depth` o si hay menos filas que `min_samples_split`. |
| L12–L13: ganancia insuficiente | Devuelve hoja sin prueba utilizable o sin `gain > 0` y `ganancia_global >= min_gain`; la igualdad sigue permitiendo dividir. |
| L15–L18: nodo y partición | Asigna Ag y umbral; `particionar_con_desconocidos` produce grupos con pesos y probabilidades. |
| L19–L20: restantes y recursión | Retira categóricos, conserva continuos y llama directamente a `_crecer_enriquecido` para cada grupo. |
| L21–L24: devolver T | Devuelve el árbol crecido; sólo la envoltura pública aplica `_podar_enriquecido` si corresponde. |

**La semejanza es estructural, no una identidad de fórmulas:** con nulos cada atributo
calcula su ganancia sobre su propia población conocida (NUL, libro pp. 28–31 / PDF pp. 2–5).
No se sustituye por `p0 - pi` usando un único p0 global. Los cortes y su referencia de todo
el entrenamiento siguen CON, libro pp. 25–26 / PDF pp. 1–2; la envoltura prepara esa
referencia una sola vez y aplica la poda únicamente después del crecimiento completo.
La poda y su aproximación aprobada siguen documentadas arriba (POD, libro pp. 37–43 / PDF pp. 1–7).

**Controles del árbol enriquecido:** `max_depth=None` no impone límite; la raíz está en
profundidad cero y `max_depth=0` devuelve su hoja mayoritaria. `min_samples_split=2`
cuenta las filas presentes en cada nodo, no su peso: la igualdad permite dividir y
no exige un mínimo de filas por hoja. Un caso desconocido repartido cuenta en cada
grupo al que llega. `min_peso_rama=2.0` sigue siendo independiente: exige peso conocido
mínimo por lado de un corte continuo, no por nodo ni para atributos categóricos.

`min_gain=1e-12` se compara con `ganancia_global = (peso_nodo / peso_raiz) * gain`.
El peso raíz es la suma original de pesos, constante durante toda la recursión;
con pesos unitarios equivale al total de filas del entrenamiento. La selección de
atributos sigue usando la ganancia local. Se conserva su corrección por desconocidos
y el requisito de ganancia positiva, incluso con `min_gain=0`. No se usa Gain Ratio
para el umbral. La profundidad y el peso original son controles internos.

Por ejemplo: `entrenar_arbol_enriquecido(D, A, max_depth=3, min_samples_split=5)`.
Los argumentos anteriores conservan su posición; los dos límites se agregan al final.
La poda se selecciona con `metodo_poda`, después del crecimiento; `podar` conserva
la compatibilidad descrita en 6.1 y `confianza` sólo afecta a la poda pesimista.


In [ ]:
def _crecer_enriquecido(filas, atributos, pesos, continuos, referencias, min_gain, minimo,
                        max_depth, min_samples_split, depth, peso_raiz, *,
                        criterio='gain', filtro_ganancia_promedio=True, min_gain_ratio=None,
                        correccion_mdl=False, min_samples_leaf=0, min_weight_leaf=0.0,
                        originales=None):
    """Integra ganancia y cortes: NUL pp. 28–31 / PDF 2–5; CON pp. 25–26 / PDF 1–2."""
    originales = [p > 0 for p in pesos] if originales is None else originales
    opciones = dict(criterio=criterio, filtro_ganancia_promedio=filtro_ganancia_promedio,
                    min_gain_ratio=min_gain_ratio, correccion_mdl=correccion_mdl,
                    min_samples_leaf=min_samples_leaf, min_weight_leaf=min_weight_leaf)
    nodo = NodoEnriquecido(filas, pesos)
    if nodo.total_weight == 0 or len(nodo.class_counts) == 1:  # L1–L2: peso vacío o clase pura.
        return nodo
    elif not atributos:  # L3–L4: hoja mayoritaria sin atributos disponibles.
        return nodo
    elif max_depth is not None and depth >= max_depth:  # La raíz tiene profundidad cero.
        return nodo
    elif len(filas) < min_samples_split:  # Cuenta filas; la igualdad permite dividir.
        return nodo
    elif info(filas, pesos) == 0:  # Conserva la parada ponderada por entropía cero.
        return nodo
    else:
        mejor = _seleccionar_prueba_enriquecida(
            filas, atributos, pesos, continuos, referencias, minimo,
            originales=originales, **opciones)  # L7–L11.
        if mejor is None:  # Sin dos ramas conocidas utilizables, no hay progreso.
            return nodo
        Ag, prueba = mejor
        ganancia_global = sum(pesos) / peso_raiz * prueba['gain']
        if not (prueba['gain'] > 0 and ganancia_global >= min_gain):  # L12–L13.
            return nodo
        else:
            nodo.attribute, nodo.threshold = Ag, prueba['threshold']  # L15.
            grupos, nodo.branch_probabilities = particionar_con_desconocidos(
                filas, Ag, pesos, nodo.threshold)  # L16: también propaga pesos fraccionarios.
            remaining = [a for a in atributos if a != Ag or a in continuos]  # L20.
            for rama, (group, pesos_grupo) in grupos.items():  # L17–L18: grupos con peso.
                nodo.children[rama] = _crecer_enriquecido(
                    group, remaining, pesos_grupo, continuos, referencias, min_gain, minimo,
                    max_depth=max_depth, min_samples_split=min_samples_split,
                    depth=depth + 1, peso_raiz=peso_raiz,
                    originales=_originales_rama_enriquecida(
                        filas, pesos, originales, Ag, nodo.threshold, rama), **opciones)  # L19–L20.
            return nodo  # L21–L24: la poda se aplica fuera del crecimiento.


def entrenar_arbol_enriquecido(filas, atributos, continuos=(), pesos=None,
                               min_gain=1e-12, min_peso_rama=2.0, confianza=0.25, podar=True,
                               max_depth=None, min_samples_split=2, *,
                               criterio='gain', filtro_ganancia_promedio=True,
                               min_gain_ratio=None, correccion_mdl=False,
                               min_samples_leaf=0, min_weight_leaf=0.0,
                               metodo_poda=None, subtree_raising=True, ccp_alpha=0.0,
                               filas_validacion=None, pesos_validacion=None):
    """Entrena sin alterar la base ni las entradas; clase en la última columna.
    continuos declara índices; min_peso_rama exige peso conocido por lado continuo.
    max_depth=None no limita; cero deja la raíz como hoja. min_samples_split cuenta filas.
    Exige gain > 0 y (peso del nodo / peso original) * gain >= min_gain.
    Conserva el ajuste por desconocidos; podar=False permite inspeccionar el árbol crecido.
    criterio compara gain ajustada o gain_ratio; el filtro promedio sólo afecta al ratio.
    min_gain_ratio acepta la prueba elegida en ambos modos; no busca otra si falla.
    MDL exige gain ajustada positiva, pero min_gain sigue usando gain crudo global.
    min_samples_leaf cuenta originales conocidos nunca repartidos; min_weight_leaf
    exige masa posterior al reparto. Ambos mínimos son restricciones de crecimiento.
    metodo_poda=None resuelve pesimista si podar=True, ninguna en otro caso.
    Reduced-error requiere validación independiente; nunca se separa implícitamente.
    Cost-complexity minimiza greedy error 0-1 ponderado crudo + ccp_alpha * hojas.
    resumen_poda guarda método efectivo y estructura antes/despues, sin reentrenar.
    Fuentes conceptuales: CON pp. 25–26 / PDF 1–2; NUL pp. 27–33 / PDF 1–7;
    POD pp. 37–43 / PDF 1–7. APIs propias y estimador aproximado, no código del libro.
    """
    _validar_minimos_hoja(min_samples_leaf, min_weight_leaf)
    if criterio not in ('gain', 'gain_ratio'):
        raise ValueError('criterio debe ser gain o gain_ratio.')
    if not isinstance(filtro_ganancia_promedio, bool) or not isinstance(correccion_mdl, bool):
        raise ValueError('filtro_ganancia_promedio y correccion_mdl deben ser booleanos.')
    if min_gain_ratio is not None and (not isfinite(min_gain_ratio) or min_gain_ratio < 0):
        raise ValueError('min_gain_ratio debe ser None o finito no negativo.')
    metodo_poda, pesos_validacion = _validar_poda_enriquecida(
        filas, podar, metodo_poda, subtree_raising, ccp_alpha, filas_validacion, pesos_validacion)
    pesos = pesos_casos(filas, pesos)
    atributos, continuos = list(atributos), set(continuos)
    referencias = {a: [fila[a] for fila in filas] for a in continuos}
    arbol = _crecer_enriquecido(
        filas, atributos, pesos, continuos, referencias, min_gain, min_peso_rama,
        max_depth=max_depth, min_samples_split=min_samples_split, depth=0, peso_raiz=sum(pesos),
        criterio=criterio, filtro_ganancia_promedio=filtro_ganancia_promedio,
        min_gain_ratio=min_gain_ratio, correccion_mdl=correccion_mdl,
        min_samples_leaf=min_samples_leaf, min_weight_leaf=min_weight_leaf)
    antes = estadisticas_arbol_enriquecido(arbol)
    if metodo_poda == 'pesimista':
        arbol = _podar_enriquecido(arbol, filas, pesos, confianza, subtree_raising)[0]
    elif metodo_poda == 'reduced_error':
        arbol = _podar_por_riesgo_enriquecido(arbol, filas_validacion, pesos_validacion)
    elif metodo_poda == 'cost_complexity':
        arbol = _podar_por_riesgo_enriquecido(arbol, filas, pesos, ccp_alpha)
    arbol.resumen_poda = {'antes': antes, 'despues': estadisticas_arbol_enriquecido(arbol),
                        'metodo': metodo_poda}
    return arbol


arbol_enriquecido = entrenar_arbol_enriquecido(D, A)  # Demostración silenciosa y separada.


## 7. Panel interactivo del árbol enriquecido

Ejecutá el notebook en Jupyter Notebook, JupyterLab, VS Code o Google Colab con un kernel Python.
Instalá `ipywidgets` en **ese mismo entorno** (`%pip install ipywidgets==8.1.8`) si falta
y reiniciá el kernel. En Colab, si no se renderizan los widgets, habilitá su gestor con
`from google.colab import output; output.enable_custom_widget_manager()`.
La última celda muestra el panel; ajustá los controles y pulsá **Entrenar**.
Cambiar un control no reentrena por sí solo. No se requiere una aplicación web externa.

| Grupo | Controles |
| --- | --- |
| **División** | `criterio`: Gain o Gain Ratio; `filtro_ganancia_promedio` sólo para Gain Ratio; umbral opcional `min_gain_ratio`; `correccion_mdl`; atributos `continuos`. |
| **Crecimiento** | `min_gain` original ponderado por el peso de la raíz; `max_depth` (raíz = 0); `min_samples_split`; `min_samples_leaf`; `min_weight_leaf`; `min_peso_rama` conocido por lado continuo. |
| **Poda** | `podar` permite/desactiva poda; `metodo_poda`; `confianza` y `subtree_raising` sólo en pesimista; `ccp_alpha` sólo en cost-complexity; filas y pesos de validación sólo en reduced-error. |
| **Resultados** | Árbol, distribución de clases, parámetros utilizados, profundidad, nodos y hojas antes y después de podar. No representa exactitud de prueba. |

Los datos actuales son categóricos: no hay continuos seleccionables y el peso por lado
y MDL quedan deshabilitados. No se convierten categorías en números.
Los mínimos por hoja arrancan en cero (desactivados) para conservar el comportamiento previo.
La casilla «Exigir Gain Ratio mínimo» permite distinguir `None` de un umbral cero.
Una opción deshabilitada no se borra: podés cambiar de método y recuperar su configuración.

### Validación independiente para reduced-error

Ingresá **JSON sin encabezado, con la clase al final** en «Validación (JSON)»,
por ejemplo `[["sunny", "hot", "high", "weak", "no"]]` para las cuatro columnas actuales.
Los pesos son un arreglo JSON del mismo largo, por ejemplo `[0.5]`; vacío significa pesos uniformes.
Podés precargarlos con:

```python
panel = crear_panel_arbol_enriquecido(
    filas, atributos, nombres, pesos=pesos,
    filas_validacion=validacion_independiente,
    pesos_validacion=pesos_de_validacion)
display(panel)
```

Estos argumentos nuevos son exclusivamente por nombre. El panel permite editar **sólo una
copia JSON de la validación**, no las matrices originales de entrenamiento.
Reduced-error sin validación genera un error claro; nunca se sustituye por entrenamiento
ni por el conjunto de prueba final. Separar correctamente esos conjuntos es responsabilidad
del usuario. Los campos de validación se ignoran en la interfaz cuando otro método está
seleccionado y no se envían al entrenador.

El resultado queda en `panel_enriquecido.arbol`, con `parametros_utilizados` y `resultados`
para inspección; no se reemplazan `arbol_enriquecido`, `D` ni `A`, ni ningún modelo original.
Un error conserva el último resultado válido. `resultados["antes"]` y `resultados["despues"]`
contienen las estadísticas estructurales, sin repetir el entrenamiento.
Para usar otras filas, atributos o pesos, pasalos al crear la fábrica; los pesos son opcionales.

Fuera de un kernel, la celda sólo define la fábrica y el impresor enriquecido:
no importa `ipywidgets`, no muestra el panel y no agrega salida.


In [ ]:
def imprimir_arbol_enriquecido(nodo, nombres, sangria=""):
    """Muestra el árbol sin importar ni entrenar el algoritmo original."""
    if nodo.attribute is None:
        print(f"{sangria}Class: {nodo.prediction}")
        return
    print(f"{sangria}{nombres[nodo.attribute]}")
    for rama, hijo in nodo.children.items():
        etiqueta = repr(rama)
        if nodo.threshold is not None:
            etiqueta = f"{'<=' if rama else '>'} {nodo.threshold:g}"
        print(f"{sangria}  [{etiqueta}]")
        imprimir_arbol_enriquecido(hijo, nombres, sangria + "    ")


def crear_panel_arbol_enriquecido(filas=D, atributos=A, nombres=columns, pesos=None, *,
                                  filas_validacion=None, pesos_validacion=None):
    """Panel local Jupyter/Colab; validación explícita editable como JSON, clase al final.
    No usa datos de prueba ni entrenamiento para completar la validación ausente.
    Conserva panel.arbol y agrega parametros_utilizados, resultados y grupos.
    """
    import json
    import ipywidgets as widgets

    estilo = {"description_width": "initial"}
    controles = {
        "criterio": widgets.Dropdown(options=[("Gain", "gain"), ("Gain Ratio", "gain_ratio")],
                                    value="gain", description="Criterio", style=estilo),
        "filtro_ganancia_promedio": widgets.Checkbox(value=True, description="Filtro de ganancia promedio"),
        "usar_min_gain_ratio": widgets.Checkbox(value=False, description="Exigir Gain Ratio mínimo"),
        "min_gain_ratio": widgets.FloatText(value=0.0, description="Gain Ratio mínimo", style=estilo),
        "correccion_mdl": widgets.Checkbox(value=False, description="Corrección MDL"),
        "min_gain": widgets.FloatText(value=1e-12, description="Ganancia mínima", style=estilo),
        "sin_limite": widgets.Checkbox(value=True, description="Sin límite de profundidad"),
        "max_depth": widgets.IntText(value=3, description="Profundidad máxima", style=estilo),
        "min_samples_split": widgets.IntText(value=2, description="Observaciones para dividir", style=estilo),
        "min_samples_leaf": widgets.IntText(value=0, description="Observaciones por hoja", style=estilo),
        "min_weight_leaf": widgets.FloatText(value=0.0, description="Peso mínimo por hoja", style=estilo),
        "min_peso_rama": widgets.FloatText(value=2.0, description="Peso conocido por lado continuo", style=estilo),
        "podar": widgets.Checkbox(value=True, description="Permitir poda"),
        "metodo_poda": widgets.Dropdown(
            options=[("Pesimista (C4.5)", "pesimista"), ("Reduced-error", "reduced_error"),
                     ("Cost-complexity (CART)", "cost_complexity"), ("Ninguna", "ninguna")],
            value="pesimista", description="Método de poda", style=estilo),
        "confianza": widgets.FloatText(value=0.25, description="Factor de confianza", style=estilo),
        "subtree_raising": widgets.Checkbox(value=True, description="Habilitar elevación de subárboles"),
        "ccp_alpha": widgets.FloatText(value=0.0, description="ccp_alpha", style=estilo),
        "filas_validacion": widgets.Textarea(
            value="" if filas_validacion is None else json.dumps(filas_validacion, ensure_ascii=False),
            description="Validación (JSON)", placeholder='[["valor", "clase"], [null, "clase"]]',
            style=estilo, layout=widgets.Layout(width="100%")),
        "pesos_validacion": widgets.Textarea(
            value="" if pesos_validacion is None else json.dumps(pesos_validacion),
            description="Pesos validación (JSON)", placeholder="Vacío: pesos uniformes",
            style=estilo, layout=widgets.Layout(width="100%")),
    }
    numericos = []
    for atributo in atributos:
        conocidos = [fila[atributo] for fila in filas if not es_desconocido(fila[atributo])]
        if conocidos and all(isinstance(v, (int, float)) and not isinstance(v, bool)
                             and isfinite(v) for v in conocidos):
            numericos.append((nombres[atributo], atributo))
    controles["continuos"] = widgets.SelectMultiple(
        options=numericos, value=(), description="Continuos", style=estilo,
        disabled=not numericos, rows=max(1, min(4, len(numericos))))
    boton = widgets.Button(description="Entrenar", icon="play", button_style="primary")
    salida = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px"))
    grupos = {}
    disposicion = {
        "División": ("criterio", "filtro_ganancia_promedio", "usar_min_gain_ratio",
                     "min_gain_ratio", "correccion_mdl", "continuos"),
        "Crecimiento": ("min_gain", "sin_limite", "max_depth", "min_samples_split",
                        "min_samples_leaf", "min_weight_leaf", "min_peso_rama"),
        "Poda": ("podar", "metodo_poda", "confianza", "subtree_raising", "ccp_alpha",
                 "filas_validacion", "pesos_validacion"),
    }
    for titulo, claves in disposicion.items():
        grupos[titulo] = widgets.VBox([widgets.HTML(f"<h4>{titulo}</h4>"),
                                      *(controles[clave] for clave in claves)])
    grupos["Poda"].children += (widgets.HTML(
        "<small>Reduced-error requiere datos independientes, sin encabezado y con clase al final. "
        "Pegá JSON o pasalos al crear el panel. Nunca uses el conjunto de prueba final.</small>"),)
    grupos["Resultados"] = widgets.VBox([widgets.HTML("<h4>Resultados</h4>"), boton, salida])
    panel = widgets.VBox([
        widgets.HTML("<h3>Parámetros del árbol enriquecido</h3>"
                     "<p>Ajustá los controles y pulsá Entrenar. Los modelos originales no cambian.</p>"),
        *grupos.values(),
    ])
    panel.controles, panel.boton_entrenar, panel.salida = controles, boton, salida
    panel.grupos, panel.arbol = grupos, None
    panel.parametros_utilizados, panel.resultados = None, None

    def sincronizar_controles(_=None):
        metodo = controles["metodo_poda"].value if controles["podar"].value else "ninguna"
        controles["max_depth"].disabled = controles["sin_limite"].value
        controles["filtro_ganancia_promedio"].disabled = controles["criterio"].value != "gain_ratio"
        controles["min_gain_ratio"].disabled = not controles["usar_min_gain_ratio"].value
        controles["min_peso_rama"].disabled = not controles["continuos"].value
        controles["correccion_mdl"].disabled = not controles["continuos"].value
        controles["metodo_poda"].disabled = not controles["podar"].value
        controles["confianza"].disabled = metodo != "pesimista"
        controles["subtree_raising"].disabled = metodo != "pesimista"
        controles["ccp_alpha"].disabled = metodo != "cost_complexity"
        controles["filas_validacion"].disabled = metodo != "reduced_error"
        controles["pesos_validacion"].disabled = metodo != "reduced_error"

    for nombre in ("sin_limite", "podar", "continuos", "criterio", "usar_min_gain_ratio", "metodo_poda"):
        controles[nombre].observe(sincronizar_controles, names="value")
    sincronizar_controles()

    def entrenar(_):
        with salida:
            salida.clear_output(wait=True)
            boton.disabled = True
            try:
                auxiliares = {"sin_limite", "usar_min_gain_ratio", "filas_validacion", "pesos_validacion"}
                opciones = {nombre: control.value for nombre, control in controles.items()
                            if nombre not in auxiliares}
                if controles["sin_limite"].value:
                    opciones["max_depth"] = None
                if not controles["usar_min_gain_ratio"].value:
                    opciones["min_gain_ratio"] = None
                if not opciones["podar"]:
                    opciones["metodo_poda"] = "ninguna"
                metodo = opciones["metodo_poda"]
                if metodo == "reduced_error":
                    texto = controles["filas_validacion"].value.strip()
                    if not texto:
                        raise ValueError("filas_validacion es obligatorio para reduced_error.")
                    opciones["filas_validacion"] = json.loads(texto)
                    texto_pesos = controles["pesos_validacion"].value.strip()
                    opciones["pesos_validacion"] = json.loads(texto_pesos) if texto_pesos else None
                for nombre in ("min_gain", "min_weight_leaf", "ccp_alpha"):
                    if not isfinite(opciones[nombre]) or opciones[nombre] < 0:
                        raise ValueError(f"{nombre} debe ser finito y >= 0.")
                if opciones["min_gain_ratio"] is not None and (
                        not isfinite(opciones["min_gain_ratio"]) or opciones["min_gain_ratio"] < 0):
                    raise ValueError("min_gain_ratio debe ser finito y >= 0.")
                if opciones["max_depth"] is not None and opciones["max_depth"] < 0:
                    raise ValueError("max_depth debe ser un entero >= 0 o no tener límite.")
                if opciones["min_samples_split"] < 2:
                    raise ValueError("min_samples_split debe ser un entero >= 2.")
                if opciones["min_samples_leaf"] < 0:
                    raise ValueError("min_samples_leaf debe ser un entero >= 0.")
                if opciones["continuos"] and (not isfinite(opciones["min_peso_rama"])
                                             or opciones["min_peso_rama"] <= 0):
                    raise ValueError("min_peso_rama debe ser finito y > 0.")
                if metodo == "pesimista" and (not isfinite(opciones["confianza"])
                                              or not 0 < opciones["confianza"] < 1):
                    raise ValueError("confianza debe ser finita y estar entre 0 y 1, sin incluirlos.")
                nuevo = entrenar_arbol_enriquecido(filas, atributos, pesos=pesos, **opciones)
                resumen = nuevo.resumen_poda
                print(f"Parámetros utilizados: {opciones}")
                for etiqueta, clave in (("Antes de podar:", "antes"), ("Después de podar:", "despues")):
                    datos = resumen[clave]
                    print(f"{etiqueta} Profundidad: {datos['profundidad']}; "
                          f"Nodos: {datos['nodos']}; Hojas: {datos['hojas']}")
                print("Árbol enriquecido entrenado:")
                imprimir_arbol_enriquecido(nuevo, nombres)
                print(f"Distribución de clases en la raíz: {nuevo.class_probabilities}")
                # Publica el resultado sólo después de completar todo el entrenamiento y la salida.
                panel.arbol, panel.parametros_utilizados = nuevo, opciones
                panel.resultados = resumen
            except (ValueError, TypeError, OverflowError) as error:
                print(f"No se pudo entrenar: {error}")
            finally:
                boton.disabled = False

    boton.on_click(entrenar)
    return panel


if "get_ipython" in globals() and getattr(get_ipython(), "kernel", None) is not None:
    from IPython.display import display
    panel_enriquecido = crear_panel_arbol_enriquecido()
    display(panel_enriquecido)
